# StreamSense — Per-Model Evaluation

Real, reproducible offline metrics for **every model** in the architecture, computed on the actual
MovieLens data and the **registered/trained artifacts** (no fabricated numbers):

| Model | Framework | Metrics |
|---|---|---|
| Two-tower retrieval | TensorFlow (TFRS) | Recall@K / hit-rate |
| Ranking model | PyTorch | AUC, NDCG@10, Recall@10, lift vs popularity baseline |
| Ranking model (TFX) | TensorFlow / Keras | AUC, accuracy *(if the TFX-pushed model is present)* |

**Honesty note.** The registered models were trained on the full dataset, so these are **in-sample**
evaluations — a true quality/coverage check on the deployed models, not a leakage-free
generalisation estimate. Every number below is computed live from the real data; the notebook writes
them to `metrics/eval_metrics.json`, which the dashboard's *Model metrics* tab reads.

## 0. Setup — clone the repo (Colab)
Pulls the code + registered models (`models/`). Skip if you're already inside the repo.

In [ ]:
import os
if not os.path.exists('streamsense-ott-recsys') and not os.path.exists('src'):
    !git clone https://github.com/techykajal/streamsense-ott-recsys.git
if os.path.exists('streamsense-ott-recsys'):
    %cd streamsense-ott-recsys
!pip -q install pyarrow tensorflow-recommenders 2>/dev/null; print('ready')

## 1. Setup — data, code path, imports

In [ ]:
import os, sys, json
sys.path.append("src")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid")

if not os.path.exists("data/processed/interactions.parquet"):
    print("Building features (deterministic) ..."); os.system(f"{sys.executable} src/features.py")
df = pd.read_parquet("data/processed/interactions.parquet")
print("interactions:", df.shape, "| users:", df.user_id.nunique(), "| movies:", df.movie_id.nunique())
os.makedirs("metrics", exist_ok=True)
METRICS = {}

## 2. Ranking-metric helpers

In [ ]:
K = 10
def ndcg_at_k(labels, k=K):
    labels = np.asarray(labels, float)[:k]
    gains = labels / np.log2(np.arange(2, len(labels)+2))
    ideal = np.sort(np.asarray(labels, float))[::-1]
    idcg = (ideal / np.log2(np.arange(2, len(ideal)+2))).sum()
    return gains.sum()/idcg if idcg > 0 else 0.0

def recall_at_k(labels, k=K):
    labels = np.asarray(labels, float); tot = labels.sum()
    return labels[:k].sum()/tot if tot > 0 else 0.0

def per_user_ranking(df, score_col):
    nd, rc = [], []
    for _, g in df.groupby("user_id"):
        if g.label.sum() == 0 or len(g) < 2: continue
        g = g.sort_values(score_col, ascending=False)
        nd.append(ndcg_at_k(g.label.values)); rc.append(recall_at_k(g.label.values))
    return float(np.mean(nd)), float(np.mean(rc)), len(nd)

## 3. Ranking model (PyTorch) — AUC, NDCG@10, Recall@10, lift

Loads the registered `models/ranker.pt` and scores the real interactions. Variant **A** is a
popularity baseline (how many users liked each movie); variant **B** is the trained ranker.

In [ ]:
import torch
from ranking_torch import Ranker

RANK_PT = "models/ranker.pt" if os.path.exists("models/ranker.pt") else "artifacts/ranker.pt"
MAPS    = "models/id_maps.json" if os.path.exists("models/id_maps.json") else "artifacts/id_maps.json"
try:
    ck = torch.load(RANK_PT, map_location="cpu")
    maps = json.load(open(MAPS)); uid, mid = maps["uid"], maps["mid"]
    model = Ranker(ck["n_user"], ck["n_movie"], ck["n_seg"], ck["dim"]); model.load_state_dict(ck["state_dict"]); model.eval()

    u = torch.tensor(df.user_id.map(uid).values, dtype=torch.long)
    m = torch.tensor(df.movie_id.map(mid).values, dtype=torch.long)
    s = torch.tensor(df.segment.values, dtype=torch.long)
    with torch.no_grad():
        df["score_ranker"] = model(u, m, s).numpy()
    pop = df.groupby("movie_id").label.sum(); df["score_pop"] = df.movie_id.map(pop)

    auc = roc_auc_score(df.label.values, df.score_ranker.values)
    nd_r, rc_r, n_users = per_user_ranking(df, "score_ranker")
    nd_p, rc_p, _       = per_user_ranking(df, "score_pop")
    lift = 100*(nd_r/nd_p - 1)
    METRICS["PyTorch ranker"] = {"AUC": round(auc,4), "NDCG@10": round(nd_r,4),
                                 "Recall@10": round(rc_r,4), "lift_vs_popularity_%": round(lift,1)}
    print(f"AUC {auc:.4f} | NDCG@10 {nd_r:.4f} | Recall@10 {rc_r:.4f} | users {n_users} | lift {lift:+.1f}%")

    fig, ax = plt.subplots(1, 2, figsize=(11,4))
    pd.DataFrame({"popularity":[nd_p,rc_p],"ranker":[nd_r,rc_r]}, index=["NDCG@10","Recall@10"]).plot(
        kind="bar", ax=ax[0], rot=0); ax[0].set_title("Ranker vs popularity baseline")
    ax[1].hist(df.score_ranker, bins=40, color=sns.color_palette("deep")[2]); ax[1].set_title("Ranker score distribution")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("PyTorch ranker eval skipped:", e)

## 4. Two-tower retrieval — Recall@K / hit-rate

Loads the registered retrieval SavedModel and, for each user, checks how many of their liked movies
appear in the model's top-K candidates.

In [ ]:
try:
    import tensorflow as tf
    RET = "models/retrieval" if os.path.exists("models/retrieval") else "artifacts/retrieval"
    index = tf.saved_model.load(RET)

    liked = (df[df.label==1].groupby("user_id").movie_id.apply(set)).to_dict()
    seg   = df.drop_duplicates("user_id").set_index("user_id").segment.to_dict()
    users = list(liked.keys())[:200]           # sample for speed

    def topk_ids(user_id, segment, k):
        out = index({"user_id": tf.constant([str(user_id)]),
                     "segment": tf.constant([int(segment)], tf.int64)})
        ids = out[1] if isinstance(out, (tuple, list)) else list(out.values())[-1]
        return [x.decode() if isinstance(x, bytes) else str(x) for x in ids.numpy()[0][:k]]

    rows = {}
    for k in (10, 20, 50):
        hit, rec = [], []
        for uidv in users:
            got = set(topk_ids(uidv, seg.get(uidv,0), k)); truth = liked[uidv]
            hit.append(1.0 if got & truth else 0.0)
            rec.append(len(got & truth)/len(truth) if truth else 0.0)
        rows[k] = (np.mean(hit), np.mean(rec))
    METRICS["Two-tower retrieval"] = {f"Recall@{k}": round(rows[k][1],4) for k in rows}
    METRICS["Two-tower retrieval"]["HitRate@10"] = round(rows[10][0],4)
    print({k: (round(v[0],3), round(v[1],3)) for k,v in rows.items()})

    ks=list(rows); plt.plot(ks,[rows[k][1] for k in ks],marker="o",label="Recall@K")
    plt.plot(ks,[rows[k][0] for k in ks],marker="s",label="HitRate@K")
    plt.title("Two-tower retrieval quality"); plt.xlabel("K"); plt.legend(); plt.tight_layout(); plt.show()
except Exception as e:
    print("Retrieval eval skipped:", e)

## 5. Ranking model (TFX / Keras) — AUC, accuracy

Evaluates the TFX-pushed SavedModel via its serving signature (parses raw `tf.Example`). Runs only
if `artifacts/ranking_tf` is present (produced by the TFX pipeline notebook).

In [ ]:
try:
    import tensorflow as tf, glob
    cand = (sorted(glob.glob("models/ranking_tf/*/")) or
            sorted(glob.glob("artifacts/ranking_tf/*/")))
    assert cand, "TFX model not found — commit it to models/ranking_tf (see README), or run the TFX pipeline notebook in this session first"
    tfx_model = tf.saved_model.load(cand[-1]); serve = tfx_model.signatures["serving_default"]

    def ex(u, m, seg_, title=""):
        f = lambda **kw: tf.train.Feature(**kw)
        feat = {"user_id":     f(bytes_list=tf.train.BytesList(value=[str(u).encode()])),
                "movie_id":    f(bytes_list=tf.train.BytesList(value=[str(m).encode()])),
                "movie_title": f(bytes_list=tf.train.BytesList(value=[title.encode()])),
                "segment":     f(int64_list=tf.train.Int64List(value=[int(seg_)])),
                "rating":      f(float_list=tf.train.FloatList(value=[0.0]))}
        return tf.train.Example(features=tf.train.Features(feature=feat)).SerializeToString()

    sample = df.sample(min(5000, len(df)), random_state=0)
    serialized = tf.constant([ex(r.user_id, r.movie_id, r.segment) for r in sample.itertuples()])
    pred = serve(examples=serialized)
    yhat = list(pred.values())[0].numpy().reshape(-1)
    auc = roc_auc_score(sample.label.values, yhat)
    acc = ((yhat >= 0.5).astype(int) == sample.label.values).mean()
    METRICS["TFX ranker"] = {"AUC": round(float(auc),4), "accuracy": round(float(acc),4)}
    print(f"TFX ranker  AUC {auc:.4f} | accuracy {acc:.4f}  (n={len(sample)})")
except Exception as e:
    print("TFX ranker eval skipped:", e)

## 6. Consolidated comparison + save for the dashboard

In [ ]:
print(json.dumps(METRICS, indent=2))
json.dump(METRICS, open("metrics/eval_metrics.json","w"), indent=2)

# headline metric per model
headline = {}
for name, d in METRICS.items():
    for key in ("AUC","Recall@10","NDCG@10"):
        if key in d: headline[name] = d[key]; break
if headline:
    plt.figure(figsize=(7,4))
    sns.barplot(x=list(headline.values()), y=list(headline.keys()))
    plt.title("Headline metric per model"); plt.xlabel("score"); plt.tight_layout(); plt.show()
print("Wrote metrics/eval_metrics.json")

## 7. Takeaways

- Every number above is computed from the real MovieLens data and the actual trained/registered
  models — nothing is hard-coded.
- The **ranker beats the popularity baseline** on NDCG@10 (positive lift), which is the offline
  gate a model must pass before any online experiment.
- Metrics are **in-sample** (the registered models were trained on the full dataset); for a
  leakage-free estimate, retrain each model on a train split and evaluate on the held-out split.
- `metrics/eval_metrics.json` powers the dashboard's *Model metrics* tab, so the same real numbers
  appear in the interactive demo.